In [27]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_sales
from src.models import SeasonalDateRegressor, SklearnRegressorConfig, SklearnRegressorWrapper
from src.pipelines.train import split_by_time
from src.evaluation import regression_metrics
from src.features import add_time_features, add_lag_features, add_rolling_features
from src.utils import prepare_submission

data_root = repo_root / "data" / "datathon-2026-round-1"

In [7]:
df = load_sales(data_root=data_root, parse_dates=True)
train_df, valid_df = split_by_time(df, date_col="Date", valid_fraction=0.2)
model = SeasonalDateRegressor()

X_train = train_df[["Date"]]
y_train = train_df["Revenue"]

model.fit(X_train, y_train)

y_pred = model.predict(valid_df[["Date"]])
y_true = valid_df["Revenue"]
metrics = regression_metrics(y_true, y_pred)
metrics

{'mae': 1644555.194360604,
 'rmse': 1965403.7315175715,
 'mape': 69.72676924136569,
 'smape': 47.24199506693533,
 'r2': -0.386503302378604}

In [29]:
# Train two models on the same feature space:
# 1) Revenue model
# 2) Ratio model for revenue_to_cogs = Revenue / COGS

df = load_sales(data_root=data_root, parse_dates=True)
df = add_time_features(df, date_col="Date")
df = add_lag_features(df, target_col="Revenue", lags=[1, 7, 14], date_col="Date")
df = add_rolling_features(df, target_col="Revenue", windows=[7, 14], date_col="Date")
df = df.dropna().reset_index(drop=True)

train_df, valid_df = split_by_time(df, date_col="Date", valid_fraction=0.2)

feature_cols = [col for col in df.columns if col not in ["Date", "Revenue", "COGS"]]
X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]

y_train_revenue = train_df["Revenue"]
y_valid_revenue = valid_df["Revenue"]

# Ratio target (guard against zero COGS in training)
eps = 1e-6
train_cogs_safe = train_df["COGS"].clip(lower=eps)
y_train_ratio = y_train_revenue / train_cogs_safe

model_config = SklearnRegressorConfig(model_type="random_forest")

# Validation-stage models
model_rev = SklearnRegressorWrapper(model_config)
model_rev.fit(X_train, y_train_revenue)
y_pred_revenue = pd.Series(model_rev.predict(X_valid), index=valid_df.index, dtype="float64")

model_ratio = SklearnRegressorWrapper(model_config)
model_ratio.fit(X_train, y_train_ratio)
y_pred_ratio = pd.Series(model_ratio.predict(X_valid), index=valid_df.index, dtype="float64")

# Reconstruct COGS from Revenue and ratio
y_pred_ratio = y_pred_ratio.clip(lower=eps)
y_pred_cogs = y_pred_revenue / y_pred_ratio

# Evaluate reconstructed targets on validation
metrics_revenue = regression_metrics(y_valid_revenue, y_pred_revenue)
metrics_cogs_recon = regression_metrics(valid_df["COGS"], y_pred_cogs)

print("Revenue metrics:", metrics_revenue)
print("Reconstructed COGS metrics:", metrics_cogs_recon)

# Retrain on all available rows for final forecasting
X_all = df[feature_cols]
y_all_revenue = df["Revenue"]
y_all_ratio = y_all_revenue / df["COGS"].clip(lower=eps)

model_rev = SklearnRegressorWrapper(model_config)
model_rev.fit(X_all, y_all_revenue)

model_ratio = SklearnRegressorWrapper(model_config)
model_ratio.fit(X_all, y_all_ratio)

Revenue metrics: {'mae': 560298.1185008506, 'rmse': 808744.6214439585, 'mape': 23.395869216465027, 'smape': 20.314012107137707, 'r2': 0.7652289065545682}
Reconstructed COGS metrics: {'mae': 513989.96614526975, 'rmse': 752954.0034694929, 'mape': 22.837494301836287, 'smape': 20.733472281211014, 'r2': 0.7301287102355447}


In [30]:
# Build final submission with recursive lag/rolling features from predicted revenue
sub_start = "2023-01-01"
sub_end = "2024-07-01"
sub_dates = pd.date_range(sub_start, sub_end, freq="D")

lags = [1, 7, 14]
windows = [7, 14]

# Fast feature assembly: one reusable NumPy row + column index lookups
feature_index = {name: idx for idx, name in enumerate(feature_cols)}
X_row = np.zeros((1, len(feature_cols)), dtype="float64")

# Keep recursive state in Python lists and incremental rolling buffers
pred_revenue_history: list[float] = []
pred_ratio_history: list[float] = []

roll_buffers: dict[int, list[float]] = {w: [] for w in windows}
roll_sums: dict[int, float] = {w: 0.0 for w in windows}
roll_sumsq: dict[int, float] = {w: 0.0 for w in windows}

for current_date in sub_dates:
    X_row.fill(0.0)

    # Time features
    if "year" in feature_index:
        X_row[0, feature_index["year"]] = float(current_date.year)
    if "month" in feature_index:
        X_row[0, feature_index["month"]] = float(current_date.month)
    if "day" in feature_index:
        X_row[0, feature_index["day"]] = float(current_date.day)
    if "day_of_week" in feature_index:
        X_row[0, feature_index["day_of_week"]] = float(current_date.dayofweek)
    if "day_of_year" in feature_index:
        X_row[0, feature_index["day_of_year"]] = float(current_date.dayofyear)
    if "week_of_year" in feature_index:
        X_row[0, feature_index["week_of_year"]] = float(current_date.isocalendar().week)
    if "is_month_end" in feature_index:
        X_row[0, feature_index["is_month_end"]] = float(int(current_date.is_month_end))
    if "is_month_start" in feature_index:
        X_row[0, feature_index["is_month_start"]] = float(int(current_date.is_month_start))
    if "is_weekend" in feature_index:
        X_row[0, feature_index["is_weekend"]] = float(int(current_date.dayofweek >= 5))

    # Lag features from previous predictions
    for lag in lags:
        col = f"Revenue_lag_{lag}"
        if col in feature_index and len(pred_revenue_history) >= lag:
            X_row[0, feature_index[col]] = pred_revenue_history[-lag]

    # Rolling features from incremental buffers
    for window in windows:
        mean_col = f"Revenue_roll_mean_{window}"
        std_col = f"Revenue_roll_std_{window}"

        if len(roll_buffers[window]) == window:
            n = float(window)
            mean_val = roll_sums[window] / n
            # Match pandas sample std (ddof=1) when window is full
            var_num = roll_sumsq[window] - (roll_sums[window] * roll_sums[window]) / n
            std_val = float(np.sqrt(max(var_num / (n - 1.0), 0.0)))

            if mean_col in feature_index:
                X_row[0, feature_index[mean_col]] = mean_val
            if std_col in feature_index:
                X_row[0, feature_index[std_col]] = std_val

    pred_rev = float(model_rev.predict(X_row)[0])
    pred_ratio = max(float(model_ratio.predict(X_row)[0]), eps)

    pred_revenue_history.append(pred_rev)
    pred_ratio_history.append(pred_ratio)

    # Update rolling buffers with current revenue prediction
    for window in windows:
        buffer = roll_buffers[window]
        buffer.append(pred_rev)
        roll_sums[window] += pred_rev
        roll_sumsq[window] += pred_rev * pred_rev

        if len(buffer) > window:
            removed = buffer.pop(0)
            roll_sums[window] -= removed
            roll_sumsq[window] -= removed * removed

sub_pred_revenue = pd.Series(pred_revenue_history, dtype="float64")
sub_pred_ratio = pd.Series(pred_ratio_history, dtype="float64")

# Reconstruct COGS from Revenue and ratio (no clipping)
sub_pred_cogs = sub_pred_revenue / sub_pred_ratio

submission = prepare_submission(
    revenue=sub_pred_revenue.values,
    cogs=sub_pred_cogs.values,
    start_date=sub_start,
    end_date=sub_end,
 )

submission.head(), submission.tail()

AttributeError: 'numpy.ndarray' object has no attribute 'reindex'

In [21]:
import os
os.makedirs(repo_root / "submissions", exist_ok=True)
submission.to_csv(repo_root / "submissions" / "baseline_submission.csv", index=False)